In [ ]:
pip install rarfile

In [ ]:
import rarfile
with rarfile.RarFile('/content/make.rar', 'r') as rf:
     rf.extractall('/content/output')


In [ ]:

with rarfile.RarFile('/content/training_resized.rar', 'r') as rf:
     rf.extractall('/content/training_resized/')

# **install libraries to radiomics**

In [ ]:
!pip install SimpleITK
!pip install radiomics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 MB 12.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 47.2 MB/s eta 0:00:00
  Created wheel for radiomics: filename=radiomics-0.1-py3-none-any.whl size=6091 sha256=172eb23f7b304ff6c02ddc803c7743a5c8c12f3e75c588ef8a61e314d47690e5
  Stored in directory: /root/.cache/pip/wheels/72/f5/9b/32efc9afd085f158d92202cb88e07ddd5793ea549381868b29
Successfully built radiomics


In [ ]:
!pip install pyradiomics --upgrade


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 29.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Discarding https://files.pythonhosted.org/packages/03/c1/20fc2c50ab1e3304da36d866042a1905a2b05a1431ece35448ab6b4578f2/pyradiomics-3.1.0.tar.gz (from https://pypi.org/simple/pyradiomics/): Requested pyradiomics from https://files.pythonhosted.org/packages/03/c1/20fc2c50ab1e3304da36d866042a1905a2b05a1431ece35448ab6b4578f2/pyradiomics-3.1.0.tar.gz has inconsistent version: expected '3.1.0', but metadata has '3.0.1a1'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 14.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.8/117.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.2/722.2 kB 37.3 MB/s eta 0:00:00
  Created wheel for pyradiomics: filename=

# **import library**

In [ ]:
import SimpleITK as sitk
from radiomics import featureextractor
import pandas as pd
import os
import numpy as np




INFO:radiomics.featureextractor:No valid config parameter, using defaults: {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True}
INFO:radiomics.featureextractor:Enabled image types: {'Original': {}}
INFO:radiomics.featureextractor:Enabled features: {'firstorder': [], 'glcm': [], 'gldm': [], 'glrlm': [], 'glszm': [], 'ngtdm': [], 'shape': []}


# **Code to feature extraction by radiomics**

In [ ]:



# إعداد المستخرج
extractor = featureextractor.RadiomicsFeatureExtractor()
extractor.settings['resampledPixelSpacing'] = None  # استخدم None لتجنب إعادة تشكيل الحجم
extractor.settings['correctMask'] = True

# إعداد المجلدات
image_folder = '/content/training_resized/training_resized'  # استبدل هذا بالمسار إلى مجلد الصور
mask_folder = '/content/output/make'  # استبدل هذا بالمسار إلى مجلد الأقنعة

# قائمة لحفظ الميزات
features_list = []
# /content/training_resized/training_resized/glioma_tumor/

# الحصول على قائمة المجلدات الفرعية
subfolders = [f for f in os.listdir(image_folder) if os.path.isdir(os.path.join(image_folder, f))]

# استخراج الميزات من كل صورة في كل مجلد فرعي، باستثناء "no_tumor"
for subfolder in subfolders:
    if subfolder == 'no_tumor':  # تخطي المجلد الذي يحمل اسم "no_tumor"
        continue

    sub_image_folder = os.path.join(image_folder, subfolder)
    sub_mask_folder = os.path.join(mask_folder, subfolder)

    image_files = [f for f in os.listdir(sub_image_folder) if f.endswith(('.jpg', '.png'))]
    mask_files = [f for f in os.listdir(sub_mask_folder) if f.endswith(('.jpg', '.png'))]

    image_files.sort()
    mask_files.sort()

    for image_file in image_files:
        mask_file = image_file  # افترض أن الأقنعة تحمل نفس اسم الصور

        if mask_file in mask_files:
            image_path = os.path.join(sub_image_folder, image_file)
            mask_path = os.path.join(sub_mask_folder, mask_file)

            # قراءة الصورة والماسك
            image = sitk.ReadImage(image_path)
            mask = sitk.ReadImage(mask_path)

            # التحقق من عدد القنوات
            pixel_id = image.GetPixelID()
            if pixel_id == sitk.sitkVectorUInt8:
                image = sitk.VectorIndexSelectionCast(image, 0)

            image = sitk.Cast(image, sitk.sitkFloat32)

            if mask.GetPixelID() != sitk.sitkUInt8:
                mask = sitk.Cast(mask, sitk.sitkUInt8)

            # استخراج الميزات
            feature_vector = extractor.execute(image, mask)
            feature_vector['Label'] = subfolder  # تعيين قيمة label إلى اسم المجلد الفرعي

            if feature_vector not in features_list:
                features_list.append(feature_vector)
        else:
            print(f"Warning: No corresponding mask found for image: {image_file}")




INFO:radiomics.featureextractor:No valid config parameter, using defaults: {'minimumROIDimensions': 2, 'minimumROISize': None, 'normalize': False, 'normalizeScale': 1, 'removeOutliers': None, 'resampledPixelSpacing': None, 'interpolator': 'sitkBSpline', 'preCrop': False, 'padDistance': 5, 'distances': [1], 'force2D': False, 'force2Ddimension': 0, 'resegmentRange': None, 'label': 1, 'additionalInfo': True}
INFO:radiomics.featureextractor:Enabled image types: {'Original': {}}
INFO:radiomics.featureextractor:Enabled features: {'firstorder': [], 'glcm': [], 'gldm': [], 'glrlm': [], 'glszm': [], 'ngtdm': [], 'shape': []}
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:Calculating features for original image
INFO:radi

INFO:radiomics.featureextractor:Computing glszm
INFO:radiomics.featureextractor:Computing ngtdm
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:Calculating features for original image
INFO:radiomics.featureextractor:Computing firstorder
INFO:radiomics.featureextractor:Computing glcm
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
INFO:radiomics.featureextractor:Computing gldm
INFO:radiomics.featureextractor:Computing glrlm
INFO:radiomics.featureextractor:Computing glszm
INFO:radiomics.featureextractor:Computing ngtdm
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
Shape features are only availa

Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:Calculating features for original image
INFO:radiomics.featureextractor:Computing firstorder
INFO:radiomics.featureextractor:Computing glcm
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
INFO:radiomics.featureextractor:Computing gldm
INFO:radiomics.featureextractor:Computing glrlm
INFO:radiomics.featureextractor:Computing glszm
INFO:radiomics.featureextractor:Computing ngtdm
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:Calculating features for original image
INF

Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:Calculating features for original image
INFO:radiomics.featureextractor:Computing firstorder
INFO:radiomics.featureextractor:Computing glcm
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
INFO:radiomics.featureextractor:Computing gldm
INFO:radiomics.featureextractor:Computing glrlm
INFO:radiomics.featureextractor:Computing glszm
INFO:radiomics.featureextractor:Computing ngtdm
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:Calculating features for original image
INF

INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:Calculating features for original image
INFO:radiomics.featureextractor:Computing firstorder
INFO:radiomics.featureextractor:Computing glcm
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
INFO:radiomics.featureextractor:Computing gldm
INFO:radiomics.featureextractor:Computing glrlm
INFO:radiomics.featureextractor:Computing glszm
INFO:radiomics.featureextractor:Computing ngtdm
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
INFO:radiomics.featureextractor:Adding 

...

INFO:radiomics.featureextractor:Computing firstorder
INFO:radiomics.featureextractor:Computing glcm
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
INFO:radiomics.featureextractor:Computing gldm
INFO:radiomics.featureextractor:Computing glrlm
INFO:radiomics.featureextractor:Computing glszm
INFO:radiomics.featureextractor:Computing ngtdm
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:Calculating features for original image
INFO:radiomics.featureextractor:Computing firstorder
INFO:radiomics.featureextractor:Computing glcm
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
INFO:radiomics.featureextractor:Computing gldm
INFO:ra

Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:Calculating features for original image
INFO:radiomics.featureextractor:Computing firstorder
INFO:radiomics.featureextractor:Computing glcm
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
INFO:radiomics.featureextractor:Computing gldm
INFO:radiomics.featureextractor:Computing glrlm
INFO:radiomics.featureextractor:Computing glszm
INFO:radiomics.featureextractor:Computing ngtdm
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:Calculating features for original image
INF

INFO:radiomics.featureextractor:Computing glrlm
INFO:radiomics.featureextractor:Computing glszm
INFO:radiomics.featureextractor:Computing ngtdm
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:Calculating features for original image
INFO:radiomics.featureextractor:Computing firstorder
INFO:radiomics.featureextractor:Computing glcm
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
INFO:radiomics.featureextractor:Computing gldm
INFO:radiomics.featureextractor:Computing glrlm
INFO:radiomics.featureextractor:Computing glszm
INFO:radiomics.featureextractor:Computing ngtdm
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loadi

INFO:radiomics.featureextractor:Computing glszm
INFO:radiomics.featureextractor:Computing ngtdm
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:Calculating features for original image
INFO:radiomics.featureextractor:Computing firstorder
INFO:radiomics.featureextractor:Computing glcm
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
INFO:radiomics.featureextractor:Computing gldm
INFO:radiomics.featureextractor:Computing glrlm
INFO:radiomics.featureextractor:Computing glszm
INFO:radiomics.featureextractor:Computing ngtdm
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
Shape features are only availa

INFO:radiomics.featureextractor:Computing glrlm
INFO:radiomics.featureextractor:Computing glszm
INFO:radiomics.featureextractor:Computing ngtdm
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loading image and mask
Shape features are only available 3D input (for 2D input, use shape2D). Found 2D input
INFO:radiomics.featureextractor:Adding image type "Original" with custom settings: {}
INFO:radiomics.featureextractor:Calculating features for original image
INFO:radiomics.featureextractor:Computing firstorder
INFO:radiomics.featureextractor:Computing glcm
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
INFO:radiomics.featureextractor:Computing gldm
INFO:radiomics.featureextractor:Computing glrlm
INFO:radiomics.featureextractor:Computing glszm
INFO:radiomics.featureextractor:Computing ngtdm
INFO:radiomics.featureextractor:Calculating features with label: 1
INFO:radiomics.featureextractor:Loadi

In [ ]:

# تحويل الميزات إلى DataFrame
features_df = pd.DataFrame(features_list)

# عرض أول 5 صفوف من DataFrame
print(features_df.head())

# حفظ الميزات إلى ملف CSV
features_df.to_csv('radiomic.csv', index=False)


  diagnostics_Versions_PyRadiomics diagnostics_Versions_Numpy  \
0                           v3.0.1                     1.26.4   
1                           v3.0.1                     1.26.4   
2                           v3.0.1                     1.26.4   
3                           v3.0.1                     1.26.4   
4                           v3.0.1                     1.26.4   

  diagnostics_Versions_SimpleITK diagnostics_Versions_PyWavelet  \
0                          2.4.0                          1.8.0   
1                          2.4.0                          1.8.0   
2                          2.4.0                          1.8.0   
3                          2.4.0                          1.8.0   
4                          2.4.0                          1.8.0   

  diagnostics_Versions_Python  \
0                     3.10.12   
1                     3.10.12   
2                     3.10.12   
3                     3.10.12   
4                     3.10.12   

       

In [ ]:
features_df.head()

,diagnostics_Versions_PyRadiomics,diagnostics_Versions_Numpy,diagnostics_Versions_SimpleITK,diagnostics_Versions_PyWavelet,diagnostics_Versions_Python,diagnostics_Configuration_Settings,diagnostics_Configuration_EnabledImageTypes,diagnostics_Image-original_Hash,diagnostics_Image-original_Dimensionality,diagnostics_Image-original_Spacing,...,original_glszm_SmallAreaLowGrayLevelEmphasis,original_glszm_ZoneEntropy,original_glszm_ZonePercentage,original_glszm_ZoneVariance,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength,Label
0,v3.0.1,1.26.4,2.4.0,1.8.0,3.10.12,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},45aaf89a4033b9b22afda32b32cee0eb4e779828,2D,"(1.0, 1.0)",...,0.10882245558566697,3.092478310944465,0.6612903225806451,1.37180249851279,0.173653715064758,0.3650637880274779,2.722251097606885,0.0225358889149967,3.8313782991202348,pituitary_tumor
1,v3.0.1,1.26.4,2.4.0,1.8.0,3.10.12,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},3d6c85a6601f34e577b1c97ef742c50744535bd1,2D,"(1.0, 1.0)",...,0.24632856341189677,3.234882667577031,0.7333333333333333,0.45867768595041325,0.8539644012944986,0.11369019422074844,9.224115651837874,0.09297286522633744,1.49126213592233,pituitary_tumor
2,v3.0.1,1.26.4,2.4.0,1.8.0,3.10.12,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},5531eab98522fac623ff1d710a1736dc7185303b,2D,"(1.0, 1.0)",...,0.1560031107103992,3.110405526805268,0.7014925373134329,1.0955183340878225,0.468905472636816,0.17771883289124668,3.2649204392239968,0.07128691583295374,1.2682253472521785,pituitary_tumor
3,v3.0.1,1.26.4,2.4.0,1.8.0,3.10.12,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},0ee46276b3cb88775d118d62bf8f6d66bf08d3d7,2D,"(1.0, 1.0)",...,0.14710416666666667,3.0337863204943627,0.7692307692307693,0.33,0.2138830162085976,0.3212520593080725,2.7391058571058564,0.01828829211551105,3.734569045412419,pituitary_tumor
4,v3.0.1,1.26.4,2.4.0,1.8.0,3.10.12,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},53fc923bf0d0beea49078f73adff2cd44b6606ac,2D,"(1.0, 1.0)",...,0.15242550021285653,2.913257072378065,0.7532467532467533,0.32372175980975026,0.44598155467720685,0.1706056129985229,2.5249631496803575,0.022953287787155212,1.3271103896103895,pituitary_tumor


In [ ]:
features_df.columns

Index(['diagnostics_Versions_PyRadiomics', 'diagnostics_Versions_Numpy',
       'diagnostics_Versions_SimpleITK', 'diagnostics_Versions_PyWavelet',
       'diagnostics_Versions_Python', 'diagnostics_Configuration_Settings',
       'diagnostics_Configuration_EnabledImageTypes',
       'diagnostics_Image-original_Hash',
       'diagnostics_Image-original_Dimensionality',
       'diagnostics_Image-original_Spacing',
       ...
       'original_glszm_SmallAreaLowGrayLevelEmphasis',
       'original_glszm_ZoneEntropy', 'original_glszm_ZonePercentage',
       'original_glszm_ZoneVariance', 'original_ngtdm_Busyness',
       'original_ngtdm_Coarseness', 'original_ngtdm_Complexity',
       'original_ngtdm_Contrast', 'original_ngtdm_Strength', 'Label'],
      dtype='object', length=116)

In [ ]:
features_df['Label'].unique()

array(['pituitary_tumor', 'meningioma_tumor', 'glioma_tumor'],
      dtype=object)

# **filtered_data_to_train.csv**

In [ ]:
filtered_df = features_df.loc[:, ~features_df.columns.str.startswith('diagnostics_')]
filtered_df.to_csv('filtered_data_to_train.csv', index=False)


In [ ]:
filtered_df.head()

,original_firstorder_10Percentile,original_firstorder_90Percentile,original_firstorder_Energy,original_firstorder_Entropy,original_firstorder_InterquartileRange,original_firstorder_Kurtosis,original_firstorder_Maximum,original_firstorder_MeanAbsoluteDeviation,original_firstorder_Mean,original_firstorder_Median,...,original_glszm_SmallAreaLowGrayLevelEmphasis,original_glszm_ZoneEntropy,original_glszm_ZonePercentage,original_glszm_ZoneVariance,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength,Label
0,9.0,94.0,268292.0,2.1473837093075816,29.5,2.620955329895057,131.0,23.132154006243496,58.45161290322581,60.0,...,0.10882245558566697,3.092478310944465,0.6612903225806451,1.37180249851279,0.173653715064758,0.3650637880274779,2.722251097606885,0.0225358889149967,3.8313782991202348,pituitary_tumor
1,20.8,111.80000000000001,265049.0,2.2534905225132977,32.0,3.1543334778898395,145.0,26.420555555555552,57.016666666666666,48.5,...,0.24632856341189677,3.234882667577031,0.7333333333333333,0.45867768595041325,0.8539644012944986,0.11369019422074844,9.224115651837874,0.09297286522633744,1.49126213592233,pituitary_tumor
2,13.0,95.8,326019.0,2.129386453088716,50.0,1.8925891501127878,119.0,27.33169971040321,62.25373134328358,70.0,...,0.1560031107103992,3.110405526805268,0.7014925373134329,1.0955183340878225,0.468905472636816,0.17771883289124668,3.2649204392239968,0.07128691583295374,1.2682253472521785,pituitary_tumor
3,40.8,103.80000000000001,332885.0,2.2012576881590538,31.0,3.004866192776751,131.0,19.33585798816568,67.27692307692308,65.0,...,0.14710416666666667,3.0337863204943627,0.7692307692307693,0.33,0.2138830162085976,0.3212520593080725,2.7391058571058564,0.01828829211551105,3.734569045412419,pituitary_tumor
4,38.2,88.0,344169.0,1.8998254204052607,31.0,2.669390436833099,120.0,16.89627255861022,63.5974025974026,64.0,...,0.15242550021285653,2.913257072378065,0.7532467532467533,0.32372175980975026,0.44598155467720685,0.1706056129985229,2.5249631496803575,0.022953287787155212,1.3271103896103895,pituitary_tumor


In [ ]:
filtered_df.columns

Index(['original_firstorder_10Percentile', 'original_firstorder_90Percentile',
       'original_firstorder_Energy', 'original_firstorder_Entropy',
       'original_firstorder_InterquartileRange',
       'original_firstorder_Kurtosis', 'original_firstorder_Maximum',
       'original_firstorder_MeanAbsoluteDeviation', 'original_firstorder_Mean',
       'original_firstorder_Median', 'original_firstorder_Minimum',
       'original_firstorder_Range',
       'original_firstorder_RobustMeanAbsoluteDeviation',
       'original_firstorder_RootMeanSquared', 'original_firstorder_Skewness',
       'original_firstorder_TotalEnergy', 'original_firstorder_Uniformity',
       'original_firstorder_Variance', 'original_glcm_Autocorrelation',
       'original_glcm_ClusterProminence', 'original_glcm_ClusterShade',
       'original_glcm_ClusterTendency', 'original_glcm_Contrast',
       'original_glcm_Correlation', 'original_glcm_DifferenceAverage',
       'original_glcm_DifferenceEntropy', 'original_glcm_

In [21]:
print(filtered_df['Label'].value_counts())

Label
pituitary_tumor     797
meningioma_tumor    776
glioma_tumor        776
Name: count, dtype: int64
